# NetlogRAG fine-tuning v3

Reproducibilan Colab orkestrator za pripremu podataka, QLoRA fine-tuning i evaluaciju triju modela. CSV datoteke učitavaju se s računala u privremeni Colab prostor, a trajni adapteri, checkpointovi i izvješća spremaju se na privatni Hugging Face Hub repozitorij.

Koristi se standardni Hugging Face sustav: Transformers, PEFT, TRL i bitsandbytes. Unsloth nije dio ovog eksperimentalnog protokola kako bi sva tri modela koristila isti trening sustav.

## 1. Repozitorij i ovisnosti

Odaberite GPU runtime. Grana je namjerno zaključana na `codex/thesis-revision`; `main` se ne mijenja.

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path("/content/RazvojIT-rijesenja")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "codex/thesis-revision",
            "https://github.com/lovro52/RazvojIT-rijesenja.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", "codex/thesis-revision"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", "codex/thesis-revision"],
        check=True,
    )
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "pull",
            "--ff-only",
            "origin",
            "codex/thesis-revision",
        ],
        check=True,
    )
os.chdir(REPO)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "experiments/requirements-colab.txt",
    ],
    check=True,
)

## 2. Model i Hugging Face

U Colabu otvorite ikonu ključa **Secrets**, dodajte tajnu naziva `HF_TOKEN` i uključite joj pristup ovom notebooku. Token mora imati dopuštenje za zapisivanje repozitorija. Nemojte ga upisivati izravno u ćeliju.

Za svaki model notebook izrađuje zaseban privatni repozitorij oblika `netlograg-<model>-thesis`.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
from huggingface_hub.utils import HfHubHTTPError

MODEL = "qwen3-1.7b"
VALID_MODELS = {"qwen3-1.7b", "smollm3-3b", "phi4-mini"}
if MODEL not in VALID_MODELS:
    raise ValueError(f"MODEL mora biti jedna od vrijednosti: {sorted(VALID_MODELS)}")

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN je prazan.")

login(token=HF_TOKEN, add_to_git_credential=False)
hub_api = HfApi(token=HF_TOKEN)
HF_USERNAME = hub_api.whoami()["name"]
HUB_PRIVATE = True
HUB_REPO_ID = f"{HF_USERNAME}/netlograg-{MODEL}-thesis"
hub_api.create_repo(
    repo_id=HUB_REPO_ID,
    repo_type="model",
    private=HUB_PRIVATE,
    exist_ok=True,
    token=HF_TOKEN,
)

INPUT_ROOT = pathlib.Path("/content/netlograg-input")
CICIDS2017_DIR = INPUT_ROOT / "CICIDS2017"
CICIDS2018_DIR = INPUT_ROOT / "CSE-CIC-IDS2018"
WORK_ROOT = pathlib.Path("/content/netlograg-work")
DATA_DIR = WORK_ROOT / "prepared-v3"
ARTIFACT_DIR = WORK_ROOT / "artifacts-v3"
REPORT_DIR = WORK_ROOT / "reports-v3"
for directory in (DATA_DIR, ARTIFACT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Hugging Face korisnik: {HF_USERNAME}")
print(f"Repozitorij rezultata: https://huggingface.co/{HUB_REPO_ID}")

## 3. Prijenos CSV datoteka s računala

Kliknite **Choose Files** i istodobno označite svih osam CICIDS2017 CSV datoteka. Možete odabrati i jednu ZIP arhivu koja ih sadrži. Datoteke se spremaju samo u privremeni Colab prostor i ne zauzimaju Google Drive.

Opcionalni CSE-CIC-IDS2018 prenosi se samo kada postavite `UPLOAD_EXTERNAL_DATASET = True`.

In [ ]:
import shutil
from zipfile import ZipFile

from google.colab import files


def _store_upload(target_dir, prompt):
    target_dir.mkdir(parents=True, exist_ok=True)
    print(prompt)
    uploaded = files.upload()
    for original_name, payload in uploaded.items():
        safe_name = pathlib.Path(original_name).name
        lower_name = safe_name.lower()
        if lower_name.endswith(".csv"):
            (target_dir / safe_name).write_bytes(payload)
        elif lower_name.endswith(".zip"):
            archive_path = INPUT_ROOT / safe_name
            archive_path.parent.mkdir(parents=True, exist_ok=True)
            archive_path.write_bytes(payload)
            with ZipFile(archive_path) as archive:
                for member in archive.infolist():
                    if member.is_dir() or not member.filename.lower().endswith(".csv"):
                        continue
                    target = target_dir / pathlib.Path(member.filename).name
                    with archive.open(member) as source, target.open("wb") as destination:
                        shutil.copyfileobj(source, destination)
            archive_path.unlink()
        else:
            print(f"Preskačem nepodržanu datoteku: {safe_name}")
    uploaded.clear()
    return sorted(target_dir.glob("*.csv"))


if INPUT_ROOT.exists():
    shutil.rmtree(INPUT_ROOT)

cicids2017_files = _store_upload(
    CICIDS2017_DIR,
    "Odaberite svih 8 CICIDS2017 CSV datoteka ili jednu ZIP arhivu.",
)
if len(cicids2017_files) != 8:
    raise RuntimeError(
        f"Očekivano je 8 CICIDS2017 CSV datoteka, pronađeno: {len(cicids2017_files)}."
    )
print("CICIDS2017 datoteke:")
for csv_path in cicids2017_files:
    print(f"  {csv_path.name}: {csv_path.stat().st_size / 1024**2:.1f} MiB")

UPLOAD_EXTERNAL_DATASET = False
if UPLOAD_EXTERNAL_DATASET:
    external_files = _store_upload(
        CICIDS2018_DIR,
        "Odaberite CSE-CIC-IDS2018 CSV datoteke ili ZIP arhivu.",
    )
    if not external_files:
        raise RuntimeError("Nije pronađena nijedna CSE-CIC-IDS2018 CSV datoteka.")
    print(f"CSE-CIC-IDS2018 datoteke: {len(external_files)}")

## 4. Zaštita rezultata i automatsko odspajanje

Nakon svake epohe Trainer šalje posljednji checkpoint na Hugging Face. Ako neka kasnija ćelija završi pogreškom, zapis pogreške pokušava se spremiti na Hub prije odspajanja. Tijekom interaktivnog ispravljanja možete privremeno postaviti `DISCONNECT_ON_ERROR = False`.

In [ ]:
import json
import time
import traceback
from datetime import UTC, datetime

from google.colab import runtime

AUTO_DISCONNECT = True
DISCONNECT_ON_ERROR = True


def _write_and_upload_status(status, *, details=None, result_files=None):
    payload = {
        "status": status,
        "model": MODEL,
        "hub_repo_id": HUB_REPO_ID,
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "files": result_files or [],
    }
    status_path = REPORT_DIR / f"{MODEL}-colab-session.json"
    status_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    hub_api.upload_file(
        path_or_fileobj=str(status_path),
        path_in_repo=f"reports/{status_path.name}",
        repo_id=HUB_REPO_ID,
        repo_type="model",
        token=HF_TOKEN,
    )
    if details:
        error_path = REPORT_DIR / f"{MODEL}-colab-error.txt"
        error_path.write_text(details, encoding="utf-8")
        hub_api.upload_file(
            path_or_fileobj=str(error_path),
            path_in_repo=f"reports/{error_path.name}",
            repo_id=HUB_REPO_ID,
            repo_type="model",
            token=HF_TOKEN,
        )


def _disconnect_after_failed_cell(result):
    error = result.error_in_exec or result.error_before_exec
    if error is None:
        return
    details = "".join(
        traceback.format_exception(type(error), error, error.__traceback__)
    )
    try:
        _write_and_upload_status("failed", details=details)
    except (HfHubHTTPError, OSError, ValueError) as upload_error:
        print(f"Nije bilo moguće spremiti zapis pogreške na Hub: {upload_error}")
    if DISCONNECT_ON_ERROR:
        time.sleep(5)
        runtime.unassign()


_ipython = get_ipython()
_previous_handler = globals().get("_AUTO_DISCONNECT_HANDLER")
if _previous_handler is not None:
    try:
        _ipython.events.unregister("post_run_cell", _previous_handler)
    except ValueError:
        pass
_AUTO_DISCONNECT_HANDLER = _disconnect_after_failed_cell
_ipython.events.register("post_run_cell", _AUTO_DISCONNECT_HANDLER)

print(
    f"Automatsko odspajanje: {AUTO_DISCONNECT}; "
    f"odspajanje nakon pogreške: {DISCONNECT_ON_ERROR}"
)

## 5. Priprema zaključanih splitova

Fingerprint grupiranje sprječava da identični vektori značajki prijeđu između treninga i testa. Izvješće sprema SHA-256 izvornih datoteka. Vanjski dataset, ako je prenesen, ne ulazi u trening.

In [ ]:
prepare_command = [
    sys.executable,
    "-m",
    "experiments.prepare_dataset",
    "--input-dir",
    str(CICIDS2017_DIR),
    "--output-dir",
    str(DATA_DIR),
    "--max-per-class",
    "5000",
    "--seed",
    "42",
]
if CICIDS2018_DIR.exists() and any(CICIDS2018_DIR.glob("*.csv")):
    prepare_command += ["--external-dir", str(CICIDS2018_DIR)]
subprocess.run(prepare_command, check=True)

## 6. QLoRA trening jednog modela

Dopušteni modeli su `qwen3-1.7b`, `smollm3-3b` i `phi4-mini`. Model je odabran u 2. odjeljku. Standardni Hugging Face Trainer nakon svake epohe šalje posljednji checkpoint u mapu `last-checkpoint` na Hubu.

Ako se raniji trening prekinuo nakon spremljene epohe, postavite `RESUME_FROM_HUB = True`.

In [ ]:
RESUME_FROM_HUB = False
resume_path = None
if RESUME_FROM_HUB:
    resume_root = ARTIFACT_DIR / MODEL / "hub-resume"
    snapshot_download(
        repo_id=HUB_REPO_ID,
        repo_type="model",
        allow_patterns=["last-checkpoint/**"],
        local_dir=str(resume_root),
        token=HF_TOKEN,
    )
    resume_path = resume_root / "last-checkpoint"
    if not (resume_path / "trainer_state.json").exists():
        raise RuntimeError("Na Hubu nije pronađen valjan last-checkpoint.")

train_command = [
    sys.executable,
    "-m",
    "experiments.finetune",
    "--model",
    MODEL,
    "--data-dir",
    str(DATA_DIR),
    "--output-dir",
    str(ARTIFACT_DIR),
    "--hub-model-id",
    HUB_REPO_ID,
    "--epochs",
    "2",
    "--seed",
    "42",
]
if HUB_PRIVATE:
    train_command.append("--hub-private-repo")
if resume_path is not None:
    train_command += ["--resume-from-checkpoint", str(resume_path)]

subprocess.run(train_command, check=True)

adapter_dir = ARTIFACT_DIR / MODEL / "adapter"
training_report = ARTIFACT_DIR / MODEL / "training_report.json"
hub_api.upload_folder(
    folder_path=str(adapter_dir),
    path_in_repo="adapter",
    repo_id=HUB_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
hub_api.upload_file(
    path_or_fileobj=str(training_report),
    path_in_repo="training_report.json",
    repo_id=HUB_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
print("Adapter i izvješće treninga spremljeni su na Hugging Face Hub.")

## 7. Usporediva base i fine-tuned evaluacija

Obje varijante koriste isti zaključani testni skup. Ne podešavajte hiperparametre prema rezultatima testnog skupa.

In [ ]:
test_file = DATA_DIR / "test.jsonl"
for variant in ("base", "fine_tuned"):
    evaluate_command = [
        sys.executable,
        "-m",
        "experiments.evaluate",
        "--model",
        MODEL,
        "--variant",
        variant,
        "--dataset",
        str(test_file),
        "--output",
        str(REPORT_DIR / f"{MODEL}-{variant}-test.json"),
        "--seed",
        "42",
    ]
    if variant == "fine_tuned":
        evaluate_command += ["--adapter", str(adapter_dir)]
    subprocess.run(evaluate_command, check=True)

## 8. Vanjski test, ako je prenesen

CSE-CIC-IDS2018 služi procjeni promjene distribucije. Zbog različitih scenarija i oznaka rezultat se izvještava zasebno.

In [ ]:
external_test = DATA_DIR / "external_test.jsonl"
if external_test.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "experiments.evaluate",
            "--model",
            MODEL,
            "--variant",
            "fine_tuned",
            "--adapter",
            str(adapter_dir),
            "--dataset",
            str(external_test),
            "--output",
            str(REPORT_DIR / f"{MODEL}-fine_tuned-external.json"),
            "--seed",
            "42",
        ],
        check=True,
    )
else:
    print("Vanjski test nije prenesen; preskačem ga.")

## 9. Spremanje rezultata i automatsko odspajanje

Ova ćelija provjerava obvezne rezultate, šalje izvješća i manifest dataseta na Hugging Face te tek nakon uspješnog prijenosa oslobađa Colab runtime. Mora ostati posljednja izvršna ćelija.

In [ ]:
expected_files = [
    DATA_DIR / "dataset_report.json",
    adapter_dir / "adapter_config.json",
    training_report,
    REPORT_DIR / f"{MODEL}-base-test.json",
    REPORT_DIR / f"{MODEL}-base-test_predictions.jsonl",
    REPORT_DIR / f"{MODEL}-fine_tuned-test.json",
    REPORT_DIR / f"{MODEL}-fine_tuned-test_predictions.jsonl",
]
if external_test.exists():
    expected_files.extend(
        [
            REPORT_DIR / f"{MODEL}-fine_tuned-external.json",
            REPORT_DIR / f"{MODEL}-fine_tuned-external_predictions.jsonl",
        ]
    )

missing_files = [str(path) for path in expected_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Nedostaju obvezne izlazne datoteke:\n" + "\n".join(missing_files)
    )

hub_api.upload_folder(
    folder_path=str(REPORT_DIR),
    path_in_repo="reports",
    repo_id=HUB_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
hub_api.upload_file(
    path_or_fileobj=str(DATA_DIR / "dataset_report.json"),
    path_in_repo="dataset/dataset_report.json",
    repo_id=HUB_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
_write_and_upload_status(
    "completed",
    result_files=[str(path) for path in expected_files],
)
print(f"Svi rezultati spremljeni su na https://huggingface.co/{HUB_REPO_ID}")

if AUTO_DISCONNECT:
    try:
        _ipython.events.unregister("post_run_cell", _AUTO_DISCONNECT_HANDLER)
    except ValueError:
        pass
    time.sleep(5)
    runtime.unassign()
else:
    print("AUTO_DISCONNECT je isključen; runtime ostaje povezan.")

## 10. Sljedeći korak: kvantizacija i lokalna brzina

Nakon treniranja sva tri modela slijedite `experiments/README.md`: spojite adapter, izvezite GGUF Q4_K_M i izmjerite stvarni Ollama runtime s `experiments.evaluate_ollama`. Sustav nazivajte real-time tek nakon definiranja ciljanog broja tokova u sekundi i usporedbe s izmjerenom propusnošću.